# Previsão de CLV (Customer Lifetime Value) com XGBoost

Neste notebook, construímos um modelo de regressão supervisionado utilizando o algoritmo **XGBoost** para prever o valor monetário total gasto por um cliente (uma representação direta de CLV) com base em seu comportamento de Recência, Frequência e o Cluster ao qual pertence.

## 1. Importando as Bibliotecas

Importamos as bibliotecas necessárias para manipulação de dados, treinamento do modelo, avaliação e persistência de arquivos.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
import joblib

print("Bibliotecas importadas com sucesso.")

Bibliotecas importadas com sucesso.


## 2. Carregando os Dados Clusterizados

Carregamos o dataset `rfm_clusters.csv` que contém os dados originais RFM acrescidos da coluna `cluster_id` gerada pelo algoritmo K-Means.

In [2]:
data_path = '../data/processed/rfm_clusters.csv'
df = pd.read_csv(data_path)

print(f"Tamanho do dataset: {df.shape}")
df.head()

Tamanho do dataset: (93357, 5)


,customer_unique_id,recency,frequency,monetary,cluster_id
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19,0
2,0000f46a3911fa3c0805444483337064,536,1,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89,1


## 3. Preparando as Variáveis (Features e Target)

Definimos as variáveis independentes ($X$) e a variável dependente ($y$):
- **Features ($X$):** `recency` (R), `frequency` (F) e `cluster_id`.
- **Target ($y$):** `monetary` (M).

In [3]:
features = ['recency', 'frequency', 'cluster_id']
target = 'monetary'

X = df[features]
y = df[target]

print("Features (X):")
print(X.head())
print("\nTarget (y):")
print(y.head())

Features (X):
   recency  frequency  cluster_id
0      111          1           0
1      114          1           0
2      536          1           1
3      320          1           1
4      287          1           1

Target (y):
0    141.90
1     27.19
2     86.22
3     43.62
4    196.89
Name: monetary, dtype: float64


## 4. Divisão em Treino e Teste

Separamos o conjunto de dados em treino e teste na proporção de **80/20**. 
Fixamos o parâmetro `random_state=42` para garantir a reprodutibilidade dos resultados.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Treinamento - Features: {X_train.shape} | Target: {y_train.shape}")
print(f"Teste - Features: {X_test.shape} | Target: {y_test.shape}")

Treinamento - Features: (74685, 3) | Target: (74685,)
Teste - Features: (18672, 3) | Target: (18672,)


## 5. Treinamento do Modelo XGBoost

Instanciamos o algoritmo `XGBRegressor` e ajustamos o modelo utilizando o conjunto de treino.

In [5]:
# Inicializando o XGBRegressor
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

print("Treinando o modelo XGBoost...")
xgb_model.fit(X_train, y_train)
print("Modelo treinado com sucesso!")

Treinando o modelo XGBoost...


Modelo treinado com sucesso!


## 6. Avaliação do Modelo

Para medir o desempenho do modelo, calculamos duas métricas principais de erro no conjunto de teste:
- **MAE (Erro Absoluto Médio):** Média das diferenças absolutas entre as previsões e os valores reais. É robusto a outliers.
- **RMSE (Erro Quadrático Médio):** Raiz quadrada da média das diferenças quadradas. Penaliza erros maiores com mais rigor.

In [6]:
# Previsões no conjunto de teste
y_pred = xgb_model.predict(X_test)

# Calculando métricas de erro
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("Avaliação do Modelo no Conjunto de Teste:")
print(f"MAE (Mean Absolute Error): R$ {mae:.2f}")
print(f"RMSE (Root Mean Squared Error): R$ {rmse:.2f}")

Avaliação do Modelo no Conjunto de Teste:
MAE (Mean Absolute Error): R$ 88.32
RMSE (Root Mean Squared Error): R$ 167.70


### Comparando Valores Reais vs Previsões

Vamos visualizar uma pequena amostra das previsões em comparação com os valores de teste reais.

In [7]:
comparison_df = pd.DataFrame({
    'Valor Real (M)': y_test,
    'Valor Previsto': y_pred,
    'Diferenca Absoluta': np.abs(y_test - y_pred)
})

comparison_df.head(15)

,Valor Real (M),Valor Previsto,Diferenca Absoluta
21568,36.69,127.751625,91.061625
75457,63.27,129.047440,65.777440
55143,78.11,139.561584,61.451584
45829,265.68,132.116013,133.563987
33285,85.79,137.993759,52.203759
69653,56.96,137.108322,80.148322
34705,51.04,140.297058,89.257058
502,98.60,134.687836,36.087836
92758,234.57,138.674728,95.895272
54260,123.74,132.778595,9.038595


## 7. Exportação do Modelo

Criamos o diretório `models/` na raiz do projeto (caso não exista) e exportamos o modelo XGBoost treinado como `xgboost_clv_model.pkl` utilizando a biblioteca `joblib`.

In [8]:
# Garantindo a existência do diretório
model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'xgboost_clv_model.pkl')

# Exportando o modelo
joblib.dump(xgb_model, model_path)
print(f"Modelo salvo com sucesso em: {model_path}")

Modelo salvo com sucesso em: ../models/xgboost_clv_model.pkl
